# Build the verified recursion-to-iteration dataset on a T4

Same script, same gate, same inputs as the local run. Only the generation
backend changes: `--backend hf` samples a whole function's attempts in one
batch on the GPU instead of one call at a time on the CPU.

| | CPU (llama-server) | T4 (batched) |
|---|---:|---:|
| per function, 6 samples | ~120 s | a few seconds |
| 582 functions | ~19 hours | ~1-2 hours |

**The GPU does not improve the yield.** Same weights, same answers. It makes
attempts cheap enough to afford more of them, and the failures are correlated -
a function the model will not de-recurse tends to stay that way - so more
samples help sublinearly. Plan on 15-25%, not 80%.

Once generation is fast, **compiling and running the candidates becomes the
bottleneck**, and that is CPU work Colab gives you two cores for. That is the
real limit on this notebook, not the model.

Runtime → Change runtime type → **T4 GPU** before running anything.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!g++ --version | head -1   # the gate needs this; Colab already has it

## 1. The code

Cloned rather than pasted, so the gate that decides which rows exist is the
same one with tests behind it. A private repo needs a token with `repo` scope.

In [ ]:
import os
import subprocess
from getpass import getpass

REPO = "safi892/fyp_training"
BRANCH = "language"
CHECKOUT = "/content/fyp"

# Out of the checkout before deleting it. A previous run left this process
# inside /content/fyp, and removing the directory you are standing in leaves the
# process with no working directory at all - git then fails with "Unable to read
# current working directory" and the clone never happens, which surfaces much
# later as a confusing FileNotFoundError on the chdir below.
os.chdir("/content")
subprocess.run(["rm", "-rf", CHECKOUT], check=True)

token = getpass("GitHub token (blank if the repo is public): ").strip()
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

# Not quiet, and the failure is read rather than assumed: a wrong token clones
# nothing and says so here instead of three cells later.
done = subprocess.run(
    ["git", "clone", "-b", BRANCH, url, CHECKOUT],
    capture_output=True, text=True,
)
if done.returncode != 0:
    # `+` binds tighter than the conditional, so the parenthesis is not optional.
    # Redacting an empty token would splice *** between every character.
    detail = done.stderr.replace(token, "***") if token else done.stderr
    raise SystemExit(f"clone failed:\n{detail}")

os.chdir(CHECKOUT)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# Kept narrow on purpose: Colab's preinstalled torch is fine and reinstalling it
# costs several minutes and sometimes a restart.
!pip install -q transformers peft accelerate
!pip list 2>/dev/null | grep -E "^(torch|transformers|peft|accelerate) "

### Is this actually on a GPU?

`torch ... +cpu` means it is not, whatever the runtime menu says, and the
batched backend on a CPU is slower than llama.cpp on the laptop. Worth ten
seconds here rather than an hour of wondering why it crawls.

In [ ]:
import torch

print("torch", torch.__version__)
if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU visible to torch.\n"
        f"  build: {torch.__version__}\n"
        "  Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart, "
        "and run from the top.\n"
        "  If it still says +cpu on a GPU runtime, the wheel is CPU-only:\n"
        "    !pip install -q torch --index-url https://download.pytorch.org/whl/cu121"
    )
print("gpu   ", torch.cuda.get_device_name(0))
print("vram  ", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## 2. The inputs, without uploading anything

Neither input has to come from the laptop.

**The functions** ship with the clone. `cleaned/merged_cleaned.jsonl` is 40 MB
and git-ignored, but the notebook never needed the corpus - only the 582
recursive functions whose signature the driver can drive. Those are extracted
once on the laptop and committed as `inputs.jsonl`, 246 KB, so `git clone`
brings them.

**The weights** download from Hugging Face. The default below is the *base*
model, `Qwen/Qwen2.5-Coder-1.5B-Instruct`, which needs no adapter and no upload.

Using the base model is not only a convenience, and it is worth reading before
assuming the fine-tune would do better. **83% of the fine-tune's own optimize
targets left the recursion in place.** It was trained on examples that tidied
recursive code rather than removing the recursion, and it reproduces that: the
CPU run measured 8%, with ten of eleven rejections never attempting the
transformation at all.

So the base model is a real candidate to propose better rewrites here, and the
short run below is what decides it. If the adapter is wanted anyway, put it on
the runtime by any means and set `ADAPTER` to its path.

In [11]:
import os

INPUTS = "/content/fyp/my_data_annotation/recursion_optimization/inputs.jsonl"
BASE = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
ADAPTER = None          # None = base model, no upload, nothing to fetch

assert os.path.exists(INPUTS), "the clone did not bring inputs.jsonl - check the branch"
count = sum(1 for line in open(INPUTS) if line.strip())
print(f"{count} recursive functions from the clone   (expected 582)")
print(f"proposer: {BASE}{' + ' + ADAPTER if ADAPTER else '  (base, no adapter)'}")

582 recursive functions from the clone   (expected 582)
proposer: Qwen/Qwen2.5-Coder-1.5B-Instruct  (base, no adapter)


## 3. Check the inputs before spending GPU time

This should print the same counts as the laptop. If it does not, the corpus
that arrived is not the corpus the numbers were measured on, and nothing below
is comparable.

In [12]:
import sys

sys.path.insert(0, "/content/fyp/src")
sys.path.insert(0, "/content/fyp/scripts")
from pathlib import Path

from build_optimize_dataset import WORDING, drivable_recursive, judge

functions = drivable_recursive(Path(INPUTS), 40)
print(f"{len(functions)} drivable recursive functions   (expected 582)")
print(f"wording: {WORDING[:60]}...")

# The gate itself, on a case whose answer is known. A wrong loop must be
# rejected here or every number this notebook produces is worthless.
rec = "int fact(int n){ if(n<=1) return 1; return n*fact(n-1); }"
itr = "int fact(int n){ int r=1; for(int i=2;i<=n;i++) r*=i; return r; }"
assert judge(rec, itr, 10.0) is None
assert judge(rec, itr.replace("r=1", "r=0"), 10.0) == "different output"
assert judge(rec, rec, 10.0) == "still recursive"
print("gate rejects a wrong rewrite and a non-rewrite: ok")

582 drivable recursive functions   (expected 582)
wording: the call stack is what makes this function work. Replace it ...
gate rejects a wrong rewrite and a non-rewrite: ok


## 4. A short run first

Twenty functions, to get a real yield and a real per-function time before
committing the session. Sixteen samples rather than six, because on a GPU they
cost about the same as one.

In [13]:
# /content, so the Contents view can reach it. Survives a kernel restart;
# does NOT survive the runtime being recycled, so download it once the short
# run has proved the yield rather than at the end of a two-hour session.
OUT = "/content/verified.jsonl"

# Built as a string rather than inlined into the ! magic: an expression with
# quotes in it does not survive IPython's {} interpolation reliably, and a
# broken command line here looks exactly like a broken model.
def build_cmd(limit, samples=16, temperature=0.9):
    adapter = f"--adapter {ADAPTER}" if ADAPTER else ""
    return (
        # PYTHONPATH, not sys.path: the pre-flight cell imported the package
        # into *this* process, and the subprocess below does not inherit that.
        f"cd /content/fyp && PYTHONPATH=src "
        f"python scripts/build_optimize_dataset.py "
        f"--backend hf --base {BASE} {adapter} "
        f"--corpus {INPUTS} --out {OUT} "
        f"--limit {limit} --samples {samples} --temperature {temperature}"
    )

print(build_cmd(20))

cd /content/fyp && PYTHONPATH=src python scripts/build_optimize_dataset.py --backend hf --base Qwen/Qwen2.5-Coder-1.5B-Instruct  --corpus /content/fyp/my_data_annotation/recursion_optimization/inputs.jsonl --out /content/verified.jsonl --limit 20 --samples 16 --temperature 0.9


In [ ]:
!{build_cmd(20)}

20 drivable recursive functions, 0 already verified, 0 already failed -> 20 to attempt
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 3.09G/3.09G [00:16<00:00, 187MB/s]
Loading weights: 100% 338/338 [00:08<00:00, 38.88it/s, Materializing param=model.norm.weight]                              
generation_config.json: 100% 242/242 [00:00<00:00, 1.10MB/s]
  [   1/20] kept    0  bad json
  [   2/20] kept    0  bad json
  [   3/20] kept    0  bad json
  [   4/20] kept    0  bad json
  [   5/20] kept    0  bad json
  [   6/20] kept    0  bad json
  [   7/20] kept    0  bad json


**Read the yield before going on.** Rows kept ÷ 20 is what 582 will give you.
At 20% that is about 115 rows; at 8% it is 47 and the GPU has bought speed
rather than a dataset. Either is a result worth writing down - the second one
says the limit is the model and not the budget.

## 5. The rest

Resumable: it skips functions already kept and already failed, so a disconnect
costs only the function in flight. Re-run this cell after a reconnect.

In [ ]:
# The rest. Resumable: it skips functions already kept and already failed, so a
# disconnect costs only the function in flight. Re-run after a reconnect.
!{build_cmd(582)}

## 6. Read what came out, then take it home

In [ ]:
import json
import os

if not os.path.exists(OUT):
    raise SystemExit(f"{OUT} does not exist - scroll up and read the run cell's traceback")

rows = [json.loads(line) for line in open(OUT) if line.strip()]
print(f"{len(rows)} verified rows\n")

# Re-checked here rather than trusted. These rows were written by a process that
# could have been interrupted mid-line, and the whole claim of this dataset is
# that every row was executed - so it is executed once more, in front of you.
for n, row in enumerate(rows, 1):
    problem = judge(row["code"], row["improved_code"], 10.0)
    print("=" * 78)
    print(f"[{n}/{len(rows)}]   re-check: {problem or 'PASSES - compiles, runs, identical output'}")
    print("-" * 78)
    print("RECURSIVE (the author's own code, unedited)")
    print(row["code"].rstrip())
    print("-" * 78)
    print("REWRITTEN (proposed by the model, kept only because it passed)")
    print(row["improved_code"].rstrip())
    print()

Same rows, as JSONL, for copy-paste:

In [ ]:
# The same rows as JSONL, to copy straight into
# my_data_annotation/recursion_optimization/verified.jsonl on the laptop.
# This output is saved inside the .ipynb, so it survives the runtime being
# recycled even if the file itself does not.
print(f"# {len(rows)} verified rows")
for row in rows:
    print(json.dumps(row, ensure_ascii=False))

And the download itself:

In [ ]:
# Three ways off the runtime. Try them in this order.
#
# 1. VS Code: the Contents view in the activity bar shows /content.
#    Right-click verified.jsonl -> Download.
#
# 2. Browser Colab: the files panel on the left, or this -
try:
    from google.colab import files

    files.download(OUT)          # no-op in some VS Code sessions
except Exception as error:
    print("files.download unavailable here:", error)
    print("use the Contents view, or the printed JSONL above")

# 3. Nothing else works: the cell above already put every row in the notebook
#    output, and the notebook is on your laptop. Copy from there.

## 7. Or push it back to the branch

Only worth it if the token you cloned with had write access. It is the
only route that leaves the result on more than one machine.


In [ ]:
import os

assert os.path.exists(OUT), f"nothing to push: {OUT} was never written"

# Route 3: commit the result to the branch from the runtime.
DEST = "my_data_annotation/recursion_optimization/verified.colab.jsonl"

!cp {OUT} /content/fyp/{DEST}
!cd /content/fyp && git config user.email "colab@local" && git config user.name "colab" \
  && git add {DEST} \
  && git commit -q -m "Verified recursion-to-iteration rows built on a Colab T4" \
  && git push -q origin {BRANCH} && echo "pushed {DEST}"